# Sequence 2 (obstacle): detection and tracking through occlusion
Same pipeline as Sequence 1 (rectify, StereoBM, median background, largest blob, Kalman filter), plus what the obstacle needs:
1. a **status** for every frame: `detected`, `partial` (robot partly hidden), `predicted` (fully hidden, prediction only), `gated` (detection rejected as an outlier),
2. a **Mahalanobis gate**, so a wrong blob cannot pull the track away after a long occlusion,
3. **no early stop** of the track during the occlusion,
4. an **RTS smoother** for the final trajectory,
5. an **evaluation** of the occlusion bridging: re-acquisition error, comparison with simple baselines, and simulated gaps in clean stretches of the sequence.

In [ ]:
import glob
import cv2
import numpy as np
from matplotlib import pyplot as plt
from stereo import load_rectification, rectify



SEQ = "2_obstacle"
DATA = f"{SEQ}/{SEQ}"   # adjust if your folders are named differently
lefts = sorted(glob.glob(f"{DATA}/left/*.png"))
rights = sorted(glob.glob(f"{DATA}/right/*.png"))
print(len(lefts), "left,", len(rights), "right")

# Same calibration as in Sequence 1
mapL, mapR, Q = load_rectification("stereo_calib.npz")

def load_pair(i):
    """Read and rectify stereo pair number i."""
    return rectify(cv2.imread(lefts[i]), cv2.imread(rights[i]), mapL, mapR)

H_img, W_img = load_pair(0)[0].shape[:2]

# Contact sheet: find out where the obstacle is and roughly when the robot is hidden
idx = np.linspace(0, len(lefts) - 1, 12).astype(int)
fig, ax = plt.subplots(3, 4, figsize=(20, 10))
for a, i in zip(ax.ravel(), idx):
    a.imshow(load_pair(i)[0][..., ::-1]); a.set_title(f"frame {i}"); a.axis("off")
plt.tight_layout(); plt.show()

# Rectification check on the middle frame: same point -> same row in both images
rectL, rectR = load_pair(len(lefts) // 2)
both = np.hstack((rectL, rectR))
for y in range(0, both.shape[0], 40):
    cv2.line(both, (0, y), (both.shape[1], y), (0, 255, 0), 1)
plt.figure(figsize=(16, 5)); plt.imshow(both[..., ::-1]); plt.axis("off")
plt.title("Rectified pair (horizontal lines must hit the same features)"); plt.show()

ModuleNotFoundError: No module named 'stereo'

In [ ]:
min_disp = 0
num_disp = 6 * 16      # must be a multiple of 16
block_size = 15        # must be odd
stereo = cv2.StereoBM_create(numDisparities=num_disp, blockSize=block_size)
stereo.setMinDisparity(min_disp)
stereo.setDisp12MaxDiff(1)
stereo.setUniquenessRatio(10)
stereo.setSpeckleRange(2)
stereo.setSpeckleWindowSize(100)

def compute_disp(rL, rR):
    """StereoBM returns disparity * 16."""
    return stereo.compute(cv2.cvtColor(rL, cv2.COLOR_BGR2GRAY),
                          cv2.cvtColor(rR, cv2.COLOR_BGR2GRAY)).astype(np.float32) / 16.0

f_px = Q[2, 3]          # focal length in pixels
B_mm = 1.0 / Q[3, 2]    # baseline in mm
print(f"f = {f_px:.1f} px, baseline = {B_mm:.1f} mm")

disp = compute_disp(rectL, rectR)
fig, ax = plt.subplots(1, 2, figsize=(18, 5))
ax[0].imshow(rectL[..., ::-1]); ax[0].set_title("Rectified left"); ax[0].axis("off")
im = ax[1].imshow(np.where(disp > 0, disp, np.nan), cmap="jet", vmin=0, vmax=60)
ax[1].set_title("StereoBM disparity (px)"); ax[1].axis("off")
plt.colorbar(im, ax=ax[1], fraction=0.025); plt.show()

In [ ]:
# 1. Background model: per-pixel MEDIAN over time (every 4th frame).
#    The obstacle is static, so it becomes part of the background. This is on purpose:
#    when the robot goes behind it, the foreground mask loses the robot (= occlusion).
bg_frames = [load_pair(i)[0] for i in range(0, len(lefts), 4)]
background = np.median(np.array(bg_frames), axis=0).astype(np.uint8)
del bg_frames

plt.figure(figsize=(10, 5)); plt.imshow(background[..., ::-1])
plt.title("Background model (temporal median)"); plt.axis("off"); plt.show()

# 2. Parameters (same as Sequence 1: re-tune if the mask looks wrong)
DIFF_THR = 25
MIN_AREA = 800
K_OPEN = np.ones((3, 3), np.uint8)
K_CLOSE = np.ones((15, 15), np.uint8)
K_MAST = cv2.getStructuringElement(cv2.MORPH_RECT, (25, 25))

def foreground_mask(img):
    diff = cv2.absdiff(img, background).max(axis=2)
    mask = (diff > DIFF_THR).astype(np.uint8) * 255
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, K_OPEN)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, K_CLOSE)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, K_MAST)
    return mask

def detect(img):
    """Largest foreground blob = robot. Returns None if nothing big enough."""
    mask = foreground_mask(img)
    n, labels, stats, cents = cv2.connectedComponentsWithStats(mask)
    if n < 2:
        return None
    k = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
    x, y, w, h, area = stats[k]
    if area < MIN_AREA:
        return None
    return {"bbox": (x, y, w, h), "centroid": cents[k], "area": area,
            "mask": labels == k, "border": x <= 1 or x + w >= img.shape[1] - 1}

# 3. Test on frames from the contact sheet. Pick some before / during / after the occlusion.
test_frames = list(idx[[1, 3, 5, 7, 9]])
fig, ax = plt.subplots(2, len(test_frames), figsize=(22, 6))
for j, i in enumerate(test_frames):
    img = load_pair(i)[0]
    det = detect(img)
    vis = img.copy()
    title = f"frame {i}: no robot"
    if det:
        x, y, w, h = det["bbox"]
        cv2.rectangle(vis, (x, y), (x + w, y + h), (0, 255, 0), 4)
        cv2.circle(vis, tuple(int(c) for c in det["centroid"]), 8, (0, 0, 255), -1)
        title = f"frame {i}: area {det['area']}" + (" (border)" if det["border"] else "")
    ax[0, j].imshow(vis[..., ::-1]); ax[0, j].set_title(title)
    ax[1, j].imshow(foreground_mask(img), cmap="gray"); ax[1, j].set_title("mask")
    ax[0, j].axis("off"); ax[1, j].axis("off")
plt.tight_layout(); plt.show()

In [ ]:
MIN_VALID = 200   # need at least this many valid disparity pixels on the robot

def locate_3d(det, disp):
    """3D position (mm): mask centroid + median disparity inside the mask."""
    d = disp[det["mask"]]
    d = d[d > 0]
    if d.size < MIN_VALID:
        return None, np.nan
    d_med = float(np.median(d))
    u, v = det["centroid"]
    X, Y, Z, W = Q @ np.array([u, v, d_med, 1.0])
    return np.array([X, Y, Z]) / W, d_med

def runs(flag):
    """Consecutive True runs of a boolean array -> list of (first, last) indices."""
    out, k, n = [], 0, len(flag)
    while k < n:
        if flag[k]:
            s = k
            while k < n and flag[k]:
                k += 1
            out.append((s, k - 1))
        else:
            k += 1
    return out

# Run detection + stereo over the whole sequence (about a minute)
results = []
for i in range(len(lefts)):
    rL, rR = load_pair(i)
    det = detect(rL)
    p3d, d_med = None, np.nan
    if det:
        p3d, d_med = locate_3d(det, compute_disp(rL, rR))
    results.append({
        "frame": i,
        "detected": det is not None,
        "border": det["border"] if det else False,
        "bbox": det["bbox"] if det else None,
        "area": float(det["area"]) if det else 0.0,
        "uv": det["centroid"] if det else np.array([np.nan, np.nan]),
        "disp": d_med,
        "xyz": p3d if p3d is not None else np.array([np.nan] * 3),
    })

n = len(results)
frames = np.arange(n)
uv = np.array([r["uv"] for r in results])
xyz = np.array([r["xyz"] for r in results]) / 1000      # metres
border = np.array([r["border"] for r in results])
area = np.array([r["area"] for r in results])
det_flag = np.array([r["detected"] for r in results])
df = frames[det_flag]
print(f"robot detected in {det_flag.sum()} frames ({df.min()}..{df.max()})")
print(f"3D position in   {(~np.isnan(xyz[:, 2])).sum()} frames")

# Frames without a detection between first and last detection = candidate occlusions
gaps = [g for g in runs(~det_flag) if g[0] > df.min() and g[1] < df.max()]
print("detection gaps inside the track (first, last):", gaps)

fig, ax = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
ax[0].plot(frames, area, ".-", ms=3); ax[0].set_ylabel("blob area (px)")
ax[0].set_title("Blob area: drops = partial occlusion, 0 = not detected"); ax[0].grid(alpha=0.3)
ax[1].plot(frames, xyz[:, 2], ".", color="tab:red"); ax[1].set_ylabel("Z (m), raw stereo")
ax[1].set_xlabel("frame"); ax[1].grid(alpha=0.3)
plt.show()

## Kalman filter
The functions are the same as in Sequence 1 (Exercise 2c). `run_tracker` below wraps the loop, so it can be rerun with hidden frames for the evaluation.

In [ ]:
def update(x, P, Z, H, R):
    y = Z - np.dot(H, x)
    S = np.dot(H, np.dot(P, H.T)) + R
    K = np.dot(P, np.dot(H.T, np.linalg.pinv(S)))
    x = x + np.dot(K, y)
    I = np.eye(P.shape[0])
    P = np.dot((I - np.dot(K, H)), P)
    return [x, P]

def predict(x, P, F, u, Q):
    x = np.dot(F, x) + u
    P = np.dot(F, np.dot(P, F.T)) + Q
    return [x, P]

def make_matrices(dim, q):
    """Constant-velocity model, state [pos_1, vel_1, pos_2, vel_2, ...]."""
    F1 = np.array([[1, 1], [0, 1]])
    H1 = np.array([[1, 0]])
    Q1 = np.array([[0.25, 0.5], [0.5, 1.0]]) * q**2
    return np.kron(np.eye(dim), F1), np.kron(np.eye(dim), H1), np.kron(np.eye(dim), Q1)

SIGMA_PX = 3.0      # centroid noise (px)
SIGMA_D = 0.25      # disparity noise (px)
BORDER = 25.0       # partly visible robot -> trust the measurement 25x less (variance)

def R_2d(bad):
    return np.eye(2) * SIGMA_PX**2 * (BORDER if bad else 1)

def R_3d(p, bad):
    Z = p[2]
    s_z = Z**2 / (f_px * B_mm) * SIGMA_D
    s_xy = SIGMA_PX * Z / f_px
    return np.diag([s_xy**2, s_xy**2, s_z**2]) * (BORDER if bad else 1)

def start_state(z, R0, vel_var):
    dim = len(z)
    x = np.zeros((2 * dim, 1)); x[0::2, 0] = z
    P = np.eye(2 * dim) * vel_var
    P[0::2, 0::2] = R0
    return x, P

# ---- new for Sequence 2 ----
CHI2_2D, CHI2_3D = 9.21, 11.34   # 99 % gate for 2 and 3 degrees of freedom
PARTIAL_RATIO = 0.6              # blob area < 60 % of the recent median -> robot partly hidden
RECENT = 15                      # frames used for the median area

def run_tracker(results, hide=(), q2=0.5, q3=5.0, max_misses=150, use_gate=True):
    """2D + 3D constant-velocity Kalman tracking with occlusion handling.
    hide: frame numbers whose measurements are removed (for simulated gaps).
    status per frame: detected / partial / predicted / gated / none."""
    hide = set(hide)
    n = len(results)
    F2, H2, Q2 = make_matrices(2, q2)
    F3, H3, Q3 = make_matrices(3, q3)
    o = dict(status=np.array(["none"] * n, dtype=object),
             uv=np.full((n, 2), np.nan), xyz=np.full((n, 3), np.nan), vel=np.full((n, 3), np.nan),
             std_uv=np.full(n, np.nan), std_xyz=np.full((n, 3), np.nan),
             prior_uv=np.full((n, 2), np.nan), prior_xyz=np.full((n, 3), np.nan),
             partial=np.zeros(n, bool), pri2={}, post2={}, pri3={}, post3={}, F2=F2, F3=F3)
    x2 = P2 = x3 = P3 = None
    miss2 = miss3 = 0
    areas = []

    for k, r in enumerate(results):
        hidden = k in hide
        z2 = r["uv"] if (r["detected"] and not hidden) else None
        z3 = r["xyz"] if (not np.isnan(r["xyz"][0]) and not hidden) else None
        recent = np.median(areas[-RECENT:]) if len(areas) >= 5 else None
        partial = z2 is not None and (r["border"] or (recent is not None and r["area"] < PARTIAL_RATIO * recent))
        o["partial"][k] = partial
        status = "none"

        # ---- 2D update (x2, P2 hold the prediction for this frame) ----
        if x2 is None:
            if z2 is not None and not partial:
                x2, P2 = start_state(z2, R_2d(False), vel_var=30**2)
                status, miss2 = "detected", 0
        else:
            o["prior_uv"][k] = x2[0::2, 0]
            o["pri2"][k] = (x2.copy(), P2.copy())
            accepted = False
            if z2 is not None:
                R = R_2d(partial)
                y = z2.reshape(-1, 1) - H2 @ x2
                S = H2 @ P2 @ H2.T + R
                d2 = (y.T @ np.linalg.inv(S) @ y).item()          # squared Mahalanobis distance
                accepted = (d2 < CHI2_2D) or not use_gate
                if not accepted:
                    status = "gated"
            if accepted:
                x2, P2 = update(x2, P2, z2.reshape(-1, 1), H2, R)
                status = "partial" if partial else "detected"
                miss2 = 0
            else:
                miss2 += 1
                if status != "gated":
                    status = "predicted"
        ok2 = status in ("detected", "partial")

        # ---- 3D update (only if the 2D measurement was accepted) ----
        use3 = z3 is not None and ok2
        if x3 is None:
            if use3 and not partial and x2 is not None:
                x3, P3 = start_state(z3, R_3d(z3, False), vel_var=200**2)
                miss3 = 0
        else:
            o["prior_xyz"][k] = x3[0::2, 0] / 1000
            o["pri3"][k] = (x3.copy(), P3.copy())
            acc3 = False
            if use3:
                R = R_3d(z3, partial)
                y = z3.reshape(-1, 1) - H3 @ x3
                S = H3 @ P3 @ H3.T + R
                d3 = (y.T @ np.linalg.inv(S) @ y).item()
                acc3 = (d3 < CHI2_3D) or not use_gate
            if acc3:
                x3, P3 = update(x3, P3, z3.reshape(-1, 1), H3, R)
                miss3 = 0
            else:
                miss3 += 1

        # ---- log, then predict the next frame ----
        if x3 is not None:
            o["xyz"][k] = x3[0::2, 0] / 1000
            o["vel"][k] = x3[1::2, 0]
            o["std_xyz"][k] = np.sqrt(np.diag(P3)[0::2]) / 1000
            o["post3"][k] = (x3.copy(), P3.copy())
            x3, P3 = predict(x3, P3, F3, np.zeros_like(x3), Q3)
            if miss3 > max_misses:
                x3 = None
        if x2 is not None:
            o["status"][k] = status
            o["uv"][k] = x2[0::2, 0]
            o["std_uv"][k] = np.sqrt(np.mean(np.diag(P2)[0::2]))
            o["post2"][k] = (x2.copy(), P2.copy())
            x2, P2 = predict(x2, P2, F2, np.zeros_like(x2), Q2)
            u_next = x2[0, 0]
            # The track ends only if the robot is lost for very long OR predicted to have left the image.
            # Being hidden behind the obstacle (predicted position still inside the image) does NOT end it.
            if miss2 > max_misses or (not ok2 and not (0 <= u_next <= W_img)):
                x2 = x3 = None
        if status == "detected":
            areas.append(r["area"])
    return o

def rts_smooth(pri, post, F):
    """Rauch-Tung-Striebel smoother over the stored priors/posteriors of one filter."""
    xs = {k: v[0].copy() for k, v in post.items()}
    Ps = {k: v[1].copy() for k, v in post.items()}
    for k in sorted(post, reverse=True):
        if (k + 1) in post and (k + 1) in pri:
            C = post[k][1] @ F.T @ np.linalg.pinv(pri[k + 1][1])
            xs[k] = post[k][0] + C @ (xs[k + 1] - pri[k + 1][0])
            Ps[k] = post[k][1] + C @ (Ps[k + 1] - pri[k + 1][1]) @ C.T
    return xs, Ps

def smooth_track(o, n):
    xs2, _ = rts_smooth(o["pri2"], o["post2"], o["F2"])
    xs3, _ = rts_smooth(o["pri3"], o["post3"], o["F3"])
    sm_uv, sm_xyz = np.full((n, 2), np.nan), np.full((n, 3), np.nan)
    for k, x in xs2.items(): sm_uv[k] = x[0::2, 0]
    for k, x in xs3.items(): sm_xyz[k] = x[0::2, 0] / 1000
    return sm_uv, sm_xyz

## Run the tracker and look at the occlusion

In [ ]:
o = run_tracker(results)
sm_uv, sm_xyz = smooth_track(o, n)
st = o["status"]
for s in ["detected", "partial", "predicted", "gated", "none"]:
    print(f"{s:10s}: {(st == s).sum()} frames")

def reacquisitions(o):
    """Occlusion events: a run of predicted/gated frames followed by an accepted measurement."""
    st, ev, k = o["status"], [], 0
    while k < len(st):
        if st[k] in ("predicted", "gated"):
            s = k
            while k < len(st) and st[k] in ("predicted", "gated"):
                k += 1
            if k < len(st) and st[k] in ("detected", "partial"):
                ev.append(dict(start=s, end=k - 1, back=k))
        else:
            k += 1
    return ev

events = reacquisitions(o)
print("\noccluded intervals (first, last, first frame back):",
      [(e["start"], e["end"], e["back"]) for e in events])

COL = {"detected": "tab:green", "partial": "orange", "predicted": "red", "gated": "magenta"}

# Plot 1: top-down trajectory coloured by status
plt.figure(figsize=(10, 5))
plt.plot(xyz[:, 0], xyz[:, 2], ".", color="lightgrey", label="stereo measurement")
for s, c in COL.items():
    m = st == s
    plt.plot(o["xyz"][m, 0], o["xyz"][m, 2], "o", ms=3, color=c, label=f"Kalman, {s}")
plt.plot(sm_xyz[:, 0], sm_xyz[:, 2], "-", color="black", lw=1, label="RTS smoothed")
plt.plot(0, 0, "k^", ms=12, label="left camera")
plt.xlabel("X (m)"); plt.ylabel("Z, depth (m)"); plt.gca().set_aspect("equal"); plt.grid(alpha=0.3)
plt.legend(fontsize=8); plt.title("Sequence 2: top-down trajectory"); plt.show()

# Plot 2: X and Z over time with hidden intervals and +-2 sigma
fig, ax = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
for a, j, name in [(ax[0], 0, "X (m)"), (ax[1], 2, "Z, depth (m)")]:
    a.plot(frames, xyz[:, j], ".", color="grey", label="stereo measurement")
    a.plot(frames, o["xyz"][:, j], "-", color="red", lw=2, label="Kalman filter")
    a.plot(frames, sm_xyz[:, j], "--", color="black", lw=1, label="RTS smoothed")
    a.fill_between(frames, o["xyz"][:, j] - 2 * o["std_xyz"][:, j], o["xyz"][:, j] + 2 * o["std_xyz"][:, j],
                   color="red", alpha=0.2, label="±2σ")
    for e in events:
        a.axvspan(e["start"], e["end"], color="blue", alpha=0.1)
    a.set_ylabel(name); a.grid(alpha=0.3)
ax[0].legend(fontsize=8); ax[1].set_xlabel("frame (blue = no accepted measurement)")
fig.suptitle("Sequence 2: 3D tracking through the occlusion"); plt.show()

# Plot 3: 2D path on the background
plt.figure(figsize=(14, 7)); plt.imshow(background[..., ::-1])
for s, c in COL.items():
    m = st == s
    plt.plot(o["uv"][m, 0], o["uv"][m, 1], "o", ms=3, color=c, label=s)
plt.legend(); plt.axis("off"); plt.title("Sequence 2: 2D path (Kalman filter, coloured by status)"); plt.show()

# Speed / distance check on frames with full measurements
good = (st == "detected") & ~np.isnan(o["vel"][:, 0])
speed = np.linalg.norm(o["vel"][good], axis=1)
print(f"speed: {speed.mean():.1f} ± {speed.std():.1f} mm/frame")
print(f"depth: {o['xyz'][good, 2].mean():.2f} ± {o['xyz'][good, 2].std():.2f} m")

## Evaluation of the occlusion bridging
There is no ground truth while the robot is hidden, so two complementary evaluations are used:
1. **Re-acquisition error (real occlusion).** Distance between the Kalman prediction and the first accepted measurement when the robot comes back. The first measurement can still be partial and therefore biased; this is flagged in the table.
2. **Simulated gaps in a clean stretch.** Measurements are hidden on purpose where the robot is fully visible, so the stereo measurement is the reference. The Kalman filter is compared with *hold last position* and a *straight-line fit* through the last 15 measurements.

In [ ]:
def baselines(results, partial, start, k_to, npre=15):
    """Predictions for frame k_to from measurements before frame `start` only. Positions in metres."""
    ks = [k for k in range(start - 1, -1, -1)
          if not np.isnan(results[k]["xyz"][0]) and not results[k]["border"] and not partial[k]][:npre][::-1]
    if len(ks) < 4:
        return None, None
    P = np.array([results[k]["xyz"] for k in ks]) / 1000
    hold = P[-1]
    line = np.array([np.polyval(np.polyfit(ks, P[:, j], 1), k_to) for j in range(3)])
    return hold, line

# ---- 1. Real occlusion: re-acquisition error ----
print("Real occlusions: error at the first accepted measurement after the gap")
print(f"{'gap (frames)':>16} {'length':>7} | {'2D KF [px]':>10} | {'3D KF':>7} {'hold':>7} {'line':>7} [mm] | first measurement")
for e in events:
    b, s = e["back"], e["start"]
    e2 = np.linalg.norm(o["prior_uv"][b] - results[b]["uv"])
    ref = results[b]["xyz"] / 1000
    row3 = ["   n/a"] * 3
    if not np.isnan(ref[0]) and not np.isnan(o["prior_xyz"][b, 0]):
        hold, line = baselines(results, o["partial"], s, b)
        errs = [np.linalg.norm(o["prior_xyz"][b] - ref)]
        if hold is not None:
            errs += [np.linalg.norm(hold - ref), np.linalg.norm(line - ref)]
        row3 = [f"{1000 * v:7.0f}" for v in errs] + ["   n/a"] * (3 - len(errs))
    print(f"{s:7d}..{e['end']:<7d} {e['end'] - s + 1:7d} | {e2:10.1f} | {row3[0]} {row3[1]} {row3[2]}        | "
          + ("partial" if o["status"][b] == "partial" else "full"))

# ---- 2. Simulated gaps in the cleanest stretch ----
clean = (~np.isnan(xyz[:, 2])) & ~border & ~o["partial"] & (st == "detected")
cr = max(runs(clean), key=lambda g: g[1] - g[0])
print(f"\ncleanest stretch: frames {cr[0]}..{cr[1]}")

def simulated_gap(start, length):
    oo = run_tracker(results, hide=range(start, start + length))
    gap = np.arange(start, start + length)
    ref = xyz[gap]
    ok = ~np.isnan(ref[:, 0]) & ~np.isnan(oo["xyz"][gap, 0])
    e_kf, e_hold, e_line = [], [], []
    for k in gap[ok]:
        hold, line = baselines(results, o["partial"], start, k)
        if hold is None:
            continue
        e_kf.append(np.linalg.norm(oo["xyz"][k] - xyz[k]))
        e_hold.append(np.linalg.norm(hold - xyz[k]))
        e_line.append(np.linalg.norm(line - xyz[k]))
    if not e_kf:
        return None
    return [1000 * np.mean(v) for v in (e_kf, e_hold, e_line)] + [1000 * e_kf[-1], 1000 * e_hold[-1], 1000 * e_line[-1]]

start = cr[0] + 20
print(f"{'gap length':>10} | mean error [mm]: {'KF':>6} {'hold':>6} {'line':>6} | error at end [mm]: {'KF':>6} {'hold':>6} {'line':>6}")
for L in [10, 20, 30, 40, 60]:
    if start + L + 5 > cr[1]:
        print(f"{L:10d} | stretch too short"); continue
    r_ = simulated_gap(start, L)
    if r_:
        print(f"{L:10d} | {'':>16}{r_[0]:6.0f} {r_[1]:6.0f} {r_[2]:6.0f} | {'':>20}{r_[3]:6.0f} {r_[4]:6.0f} {r_[5]:6.0f}")

## Video
Green box = detected, orange = partial, red = hidden (box at the Kalman estimate, circle = 2σ position uncertainty), magenta = detection rejected by the gate.

In [ ]:
FPS = 15
SCALE = 0.5
w_s, h_s = int(W_img * SCALE), int(H_img * SCALE)
writer = cv2.VideoWriter("seq2_tracking.mp4", cv2.VideoWriter_fourcc(*"mp4v"), FPS, (2 * w_s, h_s))
BGR = {"detected": (0, 255, 0), "partial": (0, 165, 255), "predicted": (0, 0, 255), "gated": (255, 0, 255)}

trail, last_wh = [], (120, 100)
for k, r in enumerate(results):
    rL, rR = load_pair(k)
    disp_k = compute_disp(rL, rR)
    vis = cv2.resize(rL, (w_s, h_s))
    s = o["status"][k]
    if r["bbox"] is not None:
        x, y, w, h = (int(v * SCALE) for v in r["bbox"])
        if s in ("detected", "partial"):
            cv2.rectangle(vis, (x, y), (x + w, y + h), BGR[s], 2)
            last_wh = (w, h)
        elif s == "gated":
            cv2.rectangle(vis, (x, y), (x + w, y + h), BGR["gated"], 1)
    if s in ("predicted", "gated") and not np.isnan(o["uv"][k, 0]):
        cx, cy = int(o["uv"][k, 0] * SCALE), int(o["uv"][k, 1] * SCALE)
        w, h = last_wh
        cv2.rectangle(vis, (cx - w // 2, cy - h // 2), (cx + w // 2, cy + h // 2), BGR["predicted"], 2)
        cv2.circle(vis, (cx, cy), int(min(2 * o["std_uv"][k] * SCALE, 300)), BGR["predicted"], 1)
        cv2.putText(vis, "HIDDEN - prediction only", (cx - w // 2, cy - h // 2 - 8),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, BGR["predicted"], 2, cv2.LINE_AA)
    if not np.isnan(o["uv"][k, 0]):
        p = (int(o["uv"][k, 0] * SCALE), int(o["uv"][k, 1] * SCALE))
        trail.append((p, BGR.get(s, (200, 200, 200))))
    for a, b in zip(trail[:-1], trail[1:]):
        cv2.line(vis, a[0], b[0], b[1], 2)

    txt = f"frame {k} [{s}]"
    if not np.isnan(o["xyz"][k, 0]):
        txt += f"   X = {o['xyz'][k, 0]:+.2f} m   Z = {o['xyz'][k, 2]:.2f} m"
    (tw, th), _ = cv2.getTextSize(txt, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 1)
    cv2.rectangle(vis, (5, 5), (15 + tw, 15 + th), (0, 0, 0), -1)
    cv2.putText(vis, txt, (10, 10 + th), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1, cv2.LINE_AA)

    d8 = np.clip(disp_k * 255 / 60, 0, 255).astype(np.uint8)
    d8s = cv2.resize(d8, (w_s, h_s), interpolation=cv2.INTER_NEAREST)
    dcol = cv2.applyColorMap(d8s, cv2.COLORMAP_JET); dcol[d8s == 0] = 0
    cv2.putText(dcol, "StereoBM disparity", (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)
    writer.write(np.hstack([vis, dcol]))
writer.release()
print("saved seq2_tracking.mp4")

In [ ]:
# Save the trajectory for the report / for comparing sequences
table = np.column_stack([frames, o["uv"], sm_uv, xyz, o["xyz"], sm_xyz, o["std_xyz"]])
np.savetxt("seq2_trajectory.csv", table, delimiter=",", fmt="%.4f",
           header="frame,u_kf,v_kf,u_smooth,v_smooth,X_meas,Y_meas,Z_meas,X_kf,Y_kf,Z_kf,X_smooth,Y_smooth,Z_smooth,sX,sY,sZ  (metres, NaN = none)")
np.save("seq2_status.npy", o["status"].astype(str))
print("saved seq2_trajectory.csv and seq2_status.npy")

# 3D map (as in Sequence 1). Use  %matplotlib qt  in a cell above to rotate it with the mouse.
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.ticker import MultipleLocator

i_empty = int(np.argmin(area))   # a frame without the robot (smallest blob), for the scene point cloud
rL0, rR0 = load_pair(i_empty)
pts0 = cv2.reprojectImageTo3D(compute_disp(rL0, rR0), Q) / 1000
d0 = compute_disp(rL0, rR0)
m0 = (d0 > 0) & (pts0[..., 2] > 0.5) & (pts0[..., 2] < 12)
cloud, rgb = pts0[m0], cv2.cvtColor(rL0, cv2.COLOR_BGR2RGB)[m0] / 255
sel = np.random.default_rng(0).choice(len(cloud), min(20000, len(cloud)), replace=False)

fig = plt.figure(figsize=(11, 8)); ax = fig.add_subplot(111, projection="3d")
ax.scatter(cloud[sel, 0], cloud[sel, 2], -cloud[sel, 1], c=rgb[sel], s=1, alpha=0.5, label="scene point cloud")
ax.plot(sm_xyz[:, 0], sm_xyz[:, 2], -sm_xyz[:, 1], "k-", lw=3, label="trajectory (RTS smoothed)")
for s, c in COL.items():
    m = st == s
    ax.scatter(o["xyz"][m, 0], o["xyz"][m, 2], -o["xyz"][m, 1], c=c, s=10, label=s)
ax.scatter(0, 0, 0, marker="^", s=150, c="black", label="left camera")
for axis in (ax.xaxis, ax.yaxis, ax.zaxis):
    axis.set_major_locator(MultipleLocator(1.0))
ax.set_xlim(-6, 6); ax.set_ylim(0, 12); ax.set_zlim(-1, 3); ax.set_box_aspect((12, 12, 4))
ax.set_xlabel("X (m)"); ax.set_ylabel("Z, depth (m)"); ax.set_zlabel("height (m)")
ax.set_title("Sequence 2: 3D map"); ax.view_init(elev=25, azim=-60); ax.legend(loc="upper left", fontsize=8)
plt.show()